# **Assignment #1 - Propositional Logic Entailment Program**

### Problem Statement
Develop a Python program that implements a simple knowledge representation and reasoning system capable of determining whether a query is entailed by a knowledge base (KB) or not. The program should build a knowledge base from a set of statements saved in a text file, then the program asks the user to input some statement, and the program then checks if it is entailed or not. If entailed, it will be added to the knowledge base. The statements (in the text file and read from the user) are human recognized ones and the program should convert them into FOL.
_______________


The program below evaluates propositional logic entailment ($KB \models \alpha$) by processing human recognized raw text knowledge base (KB) sentences and determining whether a queried sentence logically follows from them.<br>
The program uses a three-module pipeline:<br>
1. Knowledge base file reader.
2. Simple parser: Converts sentences into propositional logic Abstract Syntax Trees (ASTs).
3. Entailment checker (Reasoning Module): Uses the truth table enumeration algorithm to test entailment.<br><br>

The following figure illustrates the main components of the program and how they interact.

![Entailment System Modules](entailment_system.jpeg)

In [74]:
from IPython.display import Markdown

## **Module-1: KB File Reader**

Below is a simple function that turns a file into an array of trimmed strings.<br>
**Note:** Each line in the provided file should represent a complete KB sentence.

In [75]:
def read_kb(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        lines = [
            clean_line
            for line in file
            if (clean_line := line.strip()) and not clean_line.startswith("#")
        ]
    return lines

## **Module-2: Propositional Logic Parser**
The following python class implements a **simple** text parser that convert human sentences into propositional logic sentences
- The **input** for this parser is: human recognized raw sentences.<br>
<pre>
    eg. if today is holiday, then our services will not be available.
</pre>
- The **output** is a propositional logic abstract syntax tree representation of that sentence.<br>
<pre>
    eg.     ['IF', Q, ['NOT', P]]
            where: 
            { Q: 'today is holiday', P: 'Our services are available' }
</pre> 
<br/>

**The allowed input sentence structure is illustrated below:**

```<br>
<Expression>    := <Iff_Expr>

<Iff_Expr>      := <If_Expr> "IFF" <Iff_Expr>       <---- Lowest Precedence
                | <If_Expr>

<If_Expr>       := "IF" <Or_Expr> "THEN" <If_Expr> 
                | <Or_Expr>

<Or_Expr>       := <And_Expr> "OR" <Or_Expr> 
                | <And_Expr>

<And_Expr>      := <Not_Expr> "AND" <And_Expr> 
                | <Not_Expr>

<Not_Expr>      := "NOT" <Not_Expr> 
                | <Atomic_Expr>

<Atomic_Expr>   := "Atomic Human Sentence"          <---- Highest Precedence
```
<br/>


**Note:** All punctuation marks (e.g. ., !, ?, ,, [()]) are ignored and removed during processing.


In [ ]:
import sys
!{sys.executable} -m pip install --user -U pip
!{sys.executable} -m pip install --user spacy
!{sys.executable} -m spacy download en_core_web_sm

In [77]:
import spacy

class LogicParser:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.var_map = {}
        self.var_counter = ord('Q') 
        self.logical_ops = {"if", "then", "and", "or", "iff"}

    def _get_variable(self, prop_str):
        for var, text in self.var_map.items():
            if text == prop_str:
                return var
        var = chr(self.var_counter)
        self.var_map[var] = prop_str
        self.var_counter += 1
        return var

    def tokenize_text(self, text):
        text = text.lower()
        text = text.replace("if and only if", "iff")
        
        doc = self.nlp(text)
        tokens = []
        
        current_prop = []
        is_negated = False
        
        for token in doc:
            if token.is_punct:
                # True if the token is a punctuation mark (such as periods, commas, exclamation marks, brackets, or quotes) and False otherwise.
                continue
                
            word = token.text
            
            if word in self.logical_ops:
                if current_prop:
                    prop_str = " ".join(current_prop).strip()
                    var = self._get_variable(prop_str)
                    if is_negated:
                        tokens.extend(["NOT", var])
                    else:
                        tokens.append(var)
                    current_prop = []
                    is_negated = False
                
                if word == "implies": word = "then" 
                tokens.append(word.upper())

            # token.dep_ returns the string label of the token's syntactic dependency relation to its head (parent word) in the sentence tree, as determined by spaCy's dependency parser.    
            elif token.dep_ == "neg" or word in ["no", "not"]:
                is_negated = True

            # Boundary 3: Skip all auxiliary verbs (will, can, would, did, have)
            elif token.dep_ in ["aux", "auxpass"]:
                continue
            else:
                # token.pos_ returns the string label for the token's Part-of-Speech (POS) tag based on the Universal Dependencies schema (e.g., NOUN, VERB, ADJ).
                if token.pos_ == "VERB":
                    # returns the string representation of the token's lemma, which is its base, dictionary, or canonical form.
                    current_prop.append(token.lemma_)
                else:
                    current_prop.append(word)
                    
        if current_prop:
            prop_str = " ".join(current_prop).strip()
            var = self._get_variable(prop_str)
            if is_negated:
                tokens.extend(["NOT", var])
            else:
                tokens.append(var)
                
        return tokens

    def parse_ast(self, tokens):
        if not tokens:
            return []
        if len(tokens) == 1:
            return tokens[0]
            
        # 2. Parse IFF (Lowest Precedence: split here first)
        if 'IFF' in tokens:
            idx = tokens.index('IFF')
            left = self.parse_ast(tokens[:idx])
            right = self.parse_ast(tokens[idx+1:])
            return ['IFF', left, right]
            
        # 3. Parse IF ... THEN
        if tokens[0] == 'IF' and 'THEN' in tokens:
            then_idx = tokens.index('THEN')
            condition = self.parse_ast(tokens[1:then_idx])
            consequent = self.parse_ast(tokens[then_idx+1:])
            return ['IF', condition, consequent]
            
        # 4. Parse OR
        if 'OR' in tokens:
            idx = tokens.index('OR')
            left = self.parse_ast(tokens[:idx])
            right = self.parse_ast(tokens[idx+1:])
            return ['OR', left, right]
            
        # 5. Parse AND
        if 'AND' in tokens:
            idx = tokens.index('AND')
            left = self.parse_ast(tokens[:idx])
            right = self.parse_ast(tokens[idx+1:])
            return ['AND', left, right]
            
        # 6. Parse NOT
        if tokens[0] == 'NOT':
            return ['NOT', self.parse_ast(tokens[1:])]
            
        return tokens

    def convert(self, text):
        flat_tokens = self.tokenize_text(text)
        ast = self.parse_ast(flat_tokens)
        return ast

## **Module-3: Reasoning using Entailment**

The following function performs logical reasoning via the truth table enumeration (model-checking) algorithm. It takes the following inputs from the parser module:

- An array of parsed AST statements representing the KB.
- A parsed AST statement for the user query (Alpha).
- A map of all atomic symbols present in both the KB and Alpha.

The function returns a boolean value indicating whether **KB entails Alpha** or not.


In [78]:
def entails(KB, alpha, symbols_mappings):
    "Return True if KB entails alpha, else False. Use the truth table method."
    symbols = sorted(set([key for key in symbols_mappings.keys()]))
    unified_kb_sentence = ['AND'] + [sentence for sentence in KB]
    return check_all_models(unified_kb_sentence, alpha, symbols, {})

def check_all_models(KB, alpha, symbols, model):
    if not symbols:
        if pl_true(KB, model): 
            return pl_true(alpha, model)
        else: return True
    else:
        P, rest = symbols[0], symbols[1:]
        A = check_all_models(KB, alpha, rest, model | {P: True})
        B = check_all_models(KB , alpha, rest, model | {P: False})
        return (A and B)

def pl_true(KB, model):
    "Return True if KB is true in model, else False."
    if isinstance(KB, str): 
        return model[KB]
    op, args = KB[0], KB[1:]
    if op == 'NOT': 
        return not pl_true(args[0], model)
    elif op == 'AND': 
        return all(pl_true(arg, model) for arg in args)
    elif op == 'OR': 
        return any(pl_true(arg, model) for arg in args)
    elif op == 'IF': 
        return (not pl_true(args[0], model)) or pl_true(args[1], model)
    elif op == 'IFF': 
        return pl_true(args[0], model) == pl_true(args[1], model)
    else: print("op", op, f"Error: {op} not recognized." )


----

### Testing the Complete Pipeline Using the `security_system` Example:

* Load KB File:

In [79]:
test_kb_filepath = "kbs/security_system.txt"
test_kb_sentences = read_kb(test_kb_filepath)

print(f"\n{len(test_kb_sentences)} sentences was loaded from {test_kb_filepath} file.")
for line in test_kb_sentences:
    print('- ',line)


5 sentences was loaded from kbs/security_system.txt file.
-  The alarm sounds iff motion is detected or a window is broken
-  If there is an intruder, then motion is detected
-  If the alarm sounds, then the guard responds
-  There is an intruder
-  A window is not broken


* Parse the sentences loaded in the previous section

In [80]:
test_parser = LogicParser()
test_parsed_kb = []
for sentence in test_kb_sentences:
    ast = test_parser.convert(sentence)
    test_parsed_kb += [ast]
    print(f'- {sentence}')
    print(f"{ast}\n")

print("-------- Map --------")

test_symbols_mappings = test_parser.var_map
for (key, value) in test_symbols_mappings.items():
    print(f"{key}: {value}")


- The alarm sounds iff motion is detected or a window is broken
['IFF', 'Q', ['OR', 'R', 'S']]

- If there is an intruder, then motion is detected
['IF', 'T', 'R']

- If the alarm sounds, then the guard responds
['IF', 'Q', 'U']

- There is an intruder
T

- A window is not broken
['NOT', 'S']

-------- Map --------
Q: the alarm sound
R: motion detect
S: a window break
T: there be an intruder
U: the guard respond


* Parse the user query sentence (Alpha) and perform entailment check:

In [81]:
alphas = ["The alarm sounds", "The guard responds", "A window is broken", "There is no intruder"]
for test_alpha in alphas:
    test_alpha_ast =  test_parser.convert(test_alpha)
    print(f"\n- {test_alpha}:\t{test_alpha_ast}")
    print(f"Does KB entail '{test_alpha}':\t {entails(test_parsed_kb, test_alpha_ast, test_symbols_mappings)}")



- The alarm sounds:	Q
Does KB entail 'The alarm sounds':	 True

- The guard responds:	U
Does KB entail 'The guard responds':	 True

- A window is broken:	S
Does KB entail 'A window is broken':	 False

- There is no intruder:	['NOT', 'V']
Does KB entail 'There is no intruder':	 False


______________________________________________________________________

# **Run Entailment System:**
### ▶ Enter you knowledge base file path by running the following cell:

In [63]:
kb_filepath = input("Please provide a valid file_path for your Knowledge Base:")

if(kb_filepath):
    print(f"\nKB file_path:\t{kb_filepath}")
    parsedKB = []
    kb_sentences = read_kb(kb_filepath)

    parser = LogicParser()
    parsedKB = []
    print('\nParsed KB Sentences:\n')
    for sentence in kb_sentences:
        ast = parser.convert(sentence)
        parsedKB += [ast]
        print(f'- {sentence}')
        print(f"{ast}\n")


KB file_path:	kbs/career_path.txt

Parsed KB Sentences:

- If I pass the exam, then I graduate
['IF', 'Q', 'R']

- I get a job iff I graduate
['IFF', 'S', 'R']

- If I get a job, then I celebrate or I save money
['IF', 'S', ['OR', 'T', 'U']]

- I pass the exam
Q

- I do not save money
['NOT', 'U']



### ▶ Enter your query sentence (Alpha) by **Re-running** the following cell:

In [ ]:
alpha = input("Please provide your query sentence (Alpha):")

if(alpha):
    alpha_ast =  parser.convert(alpha)

    symbols_mappings = parser.var_map

    display(Markdown(f"## Does KB entail '{alpha}':\t **{entails(parsedKB, alpha_ast, symbols_mappings)}**"))

## Does KB entail 'I got a job':	 **True**